In [1]:
import contextlib
import joblib
import numpy as np
import pandas as pd
import pathlib
from behavioral_analysis.utility.builtin_classes.objects import load_object, save_object
from itertools import combinations, product
from joblib import Parallel, delayed
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, Perceptron, RidgeClassifier
from sklearn.metrics.pairwise import cosine_distances
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from tqdm.cli import tqdm

In [2]:
root_dir = pathlib.Path(r"H:\2026_Letter")

data_root_dir = root_dir / "data"
code_root_dir = root_dir / "code"

data_output_dir = code_root_dir / "data_output"
data_output_dir.mkdir(exist_ok=True, parents=True)

In [3]:
segmenter_order = ["scalars", "moseq"]
drug_order = [s.strip() for s in open(data_output_dir / "drug_order.txt", mode="r").readlines()]

# Figure 1

18 x 18cm

First row

    Panel a: 12 x 6
    Panel b: 6 x 6

Second row

    Panel c: 12 x 6
    Panel d: 6 x 6

Third row

    Panel e: 9 x 6
    Panel f: 9 x 6

## Panel A

## Panel B

In [4]:
approach_scalars_df = load_object(data_output_dir / "approach_scalars_df.pkl")
approach_moseq_df = load_object(data_output_dir / "approach_moseq_df.pkl")

In [5]:
def pairwise_cosine_distance(input_df):
    input_df = input_df.dropna(axis=1, how="all")
    cosine_distance_df = pd.DataFrame(data=cosine_distances(input_df), index=input_df.index.copy(), columns=input_df.index.copy())
    np.fill_diagonal(cosine_distance_df.values, np.nan)
    return cosine_distance_df

approach_scalars_cosine_distances_df = pd.concat(dict(
    raw=pairwise_cosine_distance(approach_scalars_df.loc["raw"]),
    pca=pairwise_cosine_distance(approach_scalars_df.loc["pca"])
), names=["approach"])
approach_moseq_cosine_distances_df = pd.concat(dict(
    raw=pairwise_cosine_distance(approach_moseq_df.loc["raw"]),
    pca=pairwise_cosine_distance(approach_moseq_df.loc["pca"])
), names=["approach"])
approach_moseq_cosine_distances_df

drug_class                                                                                             benzo  \
drug                                                                                              alprazolam   
highorlow                                                                                                Low   
mouse_names                                                                C57-10_.1mgkg_alprazolam_11-22-16   
approach drug_class drug       highorlow mouse_names                                                           
raw      benzo      alprazolam Low       C57-10_.1mgkg_alprazolam_11-22-16                               NaN   
                                         C57-11_.1mgkg_alprazolam_11-22-16                          0.249032   
                                         C57-12_.1mgkg_alprazolam_11-22-16                          0.119222   
                                         C57-1_.1mgkg_alprazolam_11-22-16                           0.490099   
                                         C57-2_.1mgkg_alprazolam_11-22-16                           0.473919   
...                                                                                                      ...   
pca      control    control    Low       C57-6_saline_ofa60min_110216                               0.176183   
                                         C57-7_saline_ofa60min_110216                               0.212687   
                                         C57-8_saline_ofa60min_110216                               0.210012   
                                         C57-9_saline_ofa60min_110216                               0.219878   
                                         C57-10_saline_ofa60min_110316                              0.221740   

drug_class                                                                                                    \
drug                                                                                                           
highorlow                                                                                                      
mouse_names                                                                C57-11_.1mgkg_alprazolam_11-22-16   
approach drug_class drug       highorlow mouse_names                                                           
raw      benzo      alprazolam Low       C57-10_.1mgkg_alprazolam_11-22-16                          0.249032   
                                         C57-11_.1mgkg_alprazolam_11-22-16                               NaN   
                                         C57-12_.1mgkg_alprazolam_11-22-16                          0.121835   
                                         C57-1_.1mgkg_alprazolam_11-22-16                           0.260257   
                                         C57-2_.1mgkg_alprazolam_11-22-16                           0.189311   
...                                                                                                      ...   
pca      control    control    Low       C57-6_saline_ofa60min_110216                               0.659907   
                                         C57-7_saline_ofa60min_110216                               0.413760   
                                         C57-8_saline_ofa60min_110216                               0.219809   
                                         C57-9_saline_ofa60min_110216                               0.561341   
                                         C57-10_saline_ofa60min_110316                              0.737164   

drug_class                                                                                                    \
drug                                                                                                           
highorlow                                                                                                      
mouse_names                                                                C57

In [6]:
approach_scalars_cosine_distances_df.index.names = ["approach", "drug_class1", "drug1", "dose1", "mouse_names1"]
approach_scalars_cosine_distances_df.columns.names = ["drug_class2", "drug2", "dose2", "mouse_names2"]
approach_scalars_cosine_distances_plot_df = approach_scalars_cosine_distances_df.stack(["drug_class2", "drug2", "dose2", "mouse_names2"]).rename("cosine_distance").to_frame()
_idx = approach_scalars_cosine_distances_plot_df.index.to_frame()
approach_scalars_cosine_distances_plot_df["within_or_between"] = _idx["drug1"].eq(_idx["drug2"]).apply({True: "within", False: "between"}.get)

approach_moseq_cosine_distances_df.index.names = ["approach", "drug_class1", "drug1", "dose1", "mouse_names1"]
approach_moseq_cosine_distances_df.columns.names = ["drug_class2", "drug2", "dose2", "mouse_names2"]
approach_moseq_cosine_distances_plot_df = approach_moseq_cosine_distances_df.stack(["drug_class2", "drug2", "dose2", "mouse_names2"]).rename("cosine_distance").to_frame()
_idx = approach_moseq_cosine_distances_plot_df.index.to_frame()
approach_moseq_cosine_distances_plot_df["within_or_between"] = _idx["drug1"].eq(_idx["drug2"]).apply({True: "within", False: "between"}.get)
save_object(approach_scalars_cosine_distances_plot_df, data_output_dir / "approach_scalars_cosine_distances_plot_df.pkl", overwrite=True)
save_object(approach_moseq_cosine_distances_plot_df, data_output_dir / "approach_moseq_cosine_distances_plot_df.pkl", overwrite=True)
approach_moseq_cosine_distances_plot_df

C:\Users\marti\AppData\Local\Temp\ipykernel_59704\420311946.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  approach_scalars_cosine_distances_plot_df = approach_scalars_cosine_distances_df.stack(["drug_class2", "drug2", "dose2", "mouse_names2"]).rename("cosine_distance").to_frame()
C:\Users\marti\AppData\Local\Temp\ipykernel_59704\420311946.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  approach_moseq_cosine_distances_plot_df = approach_moseq_cosine_distances_df.stack(["drug_class2", "drug2", "dose2", "mouse_names2"]).rename("cosine_distance").to_frame()


cosine_distance  \
approach drug_class1 drug1      dose1 mouse_names1                      drug_class2    drug2     dose2 mouse_names2                                        
raw      benzo       alprazolam Low   C57-10_.1mgkg_alprazolam_11-22-16 antidepressant bupropion High  C57-10_20mgkg_bupropion_11-25-16         0.619031   
                                                                                                       C57-11_20mgkg_bupropion_11-25-16         0.789784   
                                                                                                       C57-12_20mgkg_bupropion_11-25-16         0.651818   
                                                                                                       C57-1_20mgkg_bupropion_11-25-16          0.678785   
                                                                                                       C57-2_20mgkg_bupropion_11-25-16          0.655728   
...                                                                                                                                                  ...   
pca      control     control    Low   C57-10_saline_ofa60min_110316     stimulant      modafinil Low   C57-5_32mgkg_modafinil_9-17-15           1.096831   
                                                                                                       C57-6_32mgkg_modafinil_9-17-15           1.372441   
                                                                                                       C57-7_32mgkg_modafinil_9-17-15           1.329478   
                                                                                                       C57-8_32mgkg_modafinil_9-17-15           1.041109   
                                                                                                       C57-9_32mgkg_modafinil_9-17-15           1.060841   

                                                                                                                                        within_or_between  
approach drug_class1 drug1      dose1 mouse_names1                      drug_class2    drug2     dose2 mouse_names2                                        
raw      benzo       alprazolam Low   C57-10_.1mgkg_alprazolam_11-22-16 antidepressant bupropion High  C57-10_20mgkg_bupropion_11-25-16           between  
                                                                                                       C57-11_20mgkg_bupropion_11-25-16           between  
                                                                                                       C57-12_20mgkg_bupropion_11-25-16           between  
                                                                                                       C57-1_20mgkg_bupropion_11-25-16            between  
                                                                                                       C57-2_20mgkg_bupropion_11-25-16            between  
...                                                                                                                                                   ...  
pca      control     control    Low   C57-10_saline_ofa60min_110316     stimulant      modafinil Low   C57-5_32mgkg_modafinil_9-17-15             between  
                                                                                                       C57-6_32mgkg_modafinil_9-17-15             between  
                                                                                                       C57-7_32mgkg_modafinil_9-17-15             between  
                                                                                                       C57-8_32mgkg_modafinil_9-17-15             between  
                                                                                                       C57-9_32mgkg_modafinil_9-17-15             between  

[501000 rows x 2 columns]

## Panel C

## Panel D

## Panel E

## Panel F

# Figure 2

18 x 11cm

First row

    Panel a: 11 x 11
    Panel b: 7 x 11

## Panel A

In [7]:
approach_vector_df = load_object(data_output_dir / "approach_vector_df.pkl")
approach_vector_df

behavior_name                                                                          0    \
approach     segmenter phase    treatment uniform_dosage_term treatment_mouse_id             
density      moseq     Solitary bupropion high                10                  0.000000   
                                                              11                  0.000000   
                                                              12                  0.000000   
                                                              1                   0.000000   
                                                              2                   0.000000   
...                                                                                    ...   
rescaled_pca scalars   Solitary modafinil low                 5                  -4.699806   
                                                              6                   5.418398   
                                                              7                   3.897948   
                                                              8                  -4.736659   
                                                              9                  -2.074701   

behavior_name                                                                          1    \
approach     segmenter phase    treatment uniform_dosage_term treatment_mouse_id             
density      moseq     Solitary bupropion high                10                  0.000000   
                                                              11                  0.000000   
                                                              12                  0.000000   
                                                              1                   0.000000   
                                                              2                   0.000000   
...                                                                                    ...   
rescaled_pca scalars   Solitary modafinil low                 5                   3.531273   
                                                              6                  -4.798235   
                                                              7                  -2.661301   
                                                              8                   7.026998   
                                                              9                   5.624170   

behavior_name                                                                          2    \
approach     segmenter phase    treatment uniform_dosage_term treatment_mouse_id             
density      moseq     Solitary bupropion high                10                  0.000000   
                                                              11                  0.000000   
                                                              12                  0.001083   
                                                              1                   0.000000   
                                                              2                   0.000000   
...                                                                                    ...   
rescaled_pca scalars   Solitary modafinil low                 5                   7.282144   
                                                              6                   2.035397   
                                                              7                   2.688457   
                                                              8                   4.137132   
                                                              9                   0.687523   

behavior_name                                                                          3    \
approach     segmenter phase    treatment uniform_dosage_term treatment_mouse_id             
density      moseq     Solitary bupropion high                10                  0.000056   
                                                 

In [8]:
def cv_predict_closest_centroid(X, y):
    clf = NearestCentroid(metric="manhattan")
    return cross_val_predict(clf, X=X, y=y, cv=LeaveOneOut())

def vector_df_to_predicted_label_series(vector_df, prediction_func=cv_predict_closest_centroid, **kwargs):
    output_prediction_list = []
    X = vector_df.copy()
    y = X.index.to_frame()[["treatment", "uniform_dosage_term"]].apply(tuple, axis=1).astype(str)
    for (approach, segmenter, phase), app_seg_phase_X in X.groupby(["approach", "segmenter", "phase"]):
        app_seg_phase_X = app_seg_phase_X.dropna(axis=1, how="all").fillna(0)
        app_seg_phase_y = y.loc[app_seg_phase_X.index]
        output_prediction_array = prediction_func(app_seg_phase_X, app_seg_phase_y, **kwargs)
        output_prediction_series = pd.Series(output_prediction_array, index=app_seg_phase_X.index)

        output_prediction_list.append(output_prediction_series)
    return pd.concat(output_prediction_list, axis=0).apply(eval)

def vector_df_to_expected_label_series(vector_df_or_predicted_label_series):
    X = vector_df_or_predicted_label_series.copy()
    y = X.index.to_frame()[["treatment", "uniform_dosage_term"]].apply(tuple, axis=1)
    return y


In [9]:
def cv_predict_factory(clf, cv):
    if isinstance(clf, LogisticRegressionCV) or cv==1:
        def inner(X, y):
            clf.fit(X, y)
            return clf.predict(X)
    else:
        def inner(X, y):
            return cross_val_predict(clf, X=X, y=y, cv=cv)
    return inner

C_list = [0.01, 0.1, 1.0, 10.0, 100.0]
def cv_predict_logreg(X, y, C):
    clf = OneVsRestClassifier(LogisticRegression(penalty='l2', C=C, solver='liblinear', class_weight='balanced', max_iter=10000))
    pred_func = cv_predict_factory(clf, cv=LeaveOneOut())
    return pred_func(X, y)

In [10]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    # From https://stackoverflow.com/a/58936697
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [11]:
approach_segmenter_C_prediction_dict_path = data_output_dir / "approach_segmenter_C_prediction_dict.pkl"
if not approach_segmenter_C_prediction_dict_path.exists():
    save_object({}, approach_segmenter_C_prediction_dict_path, overwrite=True)


def _test_regularization(approach, segmenter, C):
    approach_segmenter_C_prediction_dict = load_object(approach_segmenter_C_prediction_dict_path)
    if (approach, segmenter, C) in approach_segmenter_C_prediction_dict:
        return
    approach_segmenter_vector_df = approach_vector_df.xs(approach, level="approach", drop_level=False).xs(segmenter, level="segmenter", drop_level=False)
    result = vector_df_to_predicted_label_series(approach_segmenter_vector_df, prediction_func=cv_predict_logreg, C=C).droplevel(["approach", "segmenter"])
    approach_segmenter_C_prediction_dict = load_object(approach_segmenter_C_prediction_dict_path)
    approach_segmenter_C_prediction_dict[(approach, segmenter, C)] = result
    save_object(approach_segmenter_C_prediction_dict, approach_segmenter_C_prediction_dict_path, overwrite=True)
    return 

C_list = np.logspace(-2, 2, 5)
total_tests = approach_vector_df.index.get_level_values("approach").nunique() * approach_vector_df.index.get_level_values("segmenter").nunique() * len(C_list)
with tqdm_joblib(tqdm(desc="Classifier application", total=total_tests)) as progress_bar:
    Parallel(n_jobs=-1)(delayed(_test_regularization)(approach, segmenter, C) for (approach, segmenter), segmenter_vector_df in approach_vector_df.groupby(["approach", "segmenter"]) for C in C_list)

approach_segmenter_C_prediction_dict = load_object(approach_segmenter_C_prediction_dict_path)
approach_segmenter_C_prediction_series = pd.concat(approach_segmenter_C_prediction_dict, names=["approach", "segmenter", "C"])
save_object(approach_segmenter_C_prediction_series, data_output_dir / "approach_segmenter_C_prediction_series.pkl", overwrite=True)
approach_segmenter_C_prediction_series

Classifier application: 100%|██████████| 40/40 [00:04<00:00,  9.95it/s]


approach  segmenter  C             phase     treatment  uniform_dosage_term  treatment_mouse_id
pca       moseq      1.000000e-02  Solitary  bupropion  high                 10                    (bupropion, very high)
                                                                             11                    (bupropion, very high)
                                                                             12                         (bupropion, high)
                                                                             1                          (bupropion, high)
                                                                             2                          (bupropion, high)
                                                                                                            ...          
density   scalars    1.000000e+12  Solitary  modafinil  low                  5                           (diazepam, high)
                                                  

In [12]:
extended_C_list = np.logspace(-12, 12, 25)
extended_tests = [(*p[0], p[1]) for p in product([("raw", "moseq"), ("raw", "scalars"), ("density", "scalars")], extended_C_list)]  # add additional C checks for raw moseq and scalars and density scalars
with tqdm_joblib(tqdm(desc="Classifier application", total=len(extended_tests))) as progress_bar:  # should be partially already covered by above analysis
    Parallel(n_jobs=-1)(delayed(_test_regularization)(approach, segmenter, C) for (approach, segmenter, C) in extended_tests)

Classifier application: 100%|██████████| 75/75 [00:01<00:00, 40.81it/s] 


In [13]:
approach_segmenter_C_prediction_dict = load_object(approach_segmenter_C_prediction_dict_path)
approach_segmenter_C_prediction_series = pd.concat(approach_segmenter_C_prediction_dict, names=["approach", "segmenter", "C"])
save_object(approach_segmenter_C_prediction_series, data_output_dir / "approach_segmenter_C_prediction_series.pkl", overwrite=True)
approach_segmenter_C_prediction_series

approach  segmenter  C             phase     treatment  uniform_dosage_term  treatment_mouse_id
pca       moseq      1.000000e-02  Solitary  bupropion  high                 10                    (bupropion, very high)
                                                                             11                    (bupropion, very high)
                                                                             12                         (bupropion, high)
                                                                             1                          (bupropion, high)
                                                                             2                          (bupropion, high)
                                                                                                            ...          
density   scalars    1.000000e+12  Solitary  modafinil  low                  5                           (diazepam, high)
                                                  

In [14]:
approach_segmenter_C_prediction_series = load_object(data_output_dir / "approach_segmenter_C_prediction_series.pkl")
approach_segmenter_C_prediction_series

approach  segmenter  C             phase     treatment  uniform_dosage_term  treatment_mouse_id
pca       moseq      1.000000e-02  Solitary  bupropion  high                 10                    (bupropion, very high)
                                                                             11                    (bupropion, very high)
                                                                             12                         (bupropion, high)
                                                                             1                          (bupropion, high)
                                                                             2                          (bupropion, high)
                                                                                                            ...          
density   scalars    1.000000e+12  Solitary  modafinil  low                  5                           (diazepam, high)
                                                  

In [15]:
app_seg_c_global_recall_series_dict = {}
reduced_segmenter_C_prediction_series = approach_segmenter_C_prediction_series.sort_index()
for (approach, segmenter, c), sub_predicted_label_series in reduced_segmenter_C_prediction_series.groupby(["approach", "segmenter", "C"]):
    sub_predicted_label_series = sub_predicted_label_series.droplevel(["approach", "C"])
    expected_label_series = vector_df_to_expected_label_series(sub_predicted_label_series)
    global_recall = sub_predicted_label_series.eq(expected_label_series).value_counts(normalize=True)[True]
    app_seg_c_global_recall_series_dict[(approach, segmenter, c)] = global_recall
app_seg_c_global_recall_series = pd.Series(app_seg_c_global_recall_series_dict, name="global_recall")
app_seg_c_global_recall_series.index.names = ["approach", "segmenter", "C"]
app_seg_c_global_recall_series

approach      segmenter  C     
density       moseq      0.01      0.528942
                         0.10      0.552894
                         1.00      0.546906
                         10.00     0.550898
                         100.00    0.578842
                                     ...   
rescaled_pca  scalars    0.01      0.441118
                         0.10      0.419162
                         1.00      0.389222
                         10.00     0.359281
                         100.00    0.347305
Name: global_recall, Length: 100, dtype: float64

In [16]:
optimal_c_series = app_seg_c_global_recall_series.loc[pd.IndexSlice[:, :, 0.01:100]  # filter here for actual area observed in publication
                                                      ].groupby(["approach", "segmenter"]).apply(lambda x: x.droplevel(["approach", "segmenter"]).idxmax())
optimal_c_series

approach      segmenter
density       moseq        100.00
              scalars      100.00
pca           moseq        100.00
              scalars       10.00
raw           moseq        100.00
              scalars        0.01
rescaled_pca  moseq          0.01
              scalars        0.01
Name: global_recall, dtype: float64

In [17]:
save_object(app_seg_c_global_recall_series, data_output_dir / "app_seg_c_global_recall_series.pkl", overwrite=True)
save_object(optimal_c_series, data_output_dir / "optimal_c_series.pkl", overwrite=True)

In [18]:
classifier_store_dict = {
    "LDA": LinearDiscriminantAnalysis(),
    "NC_Manhattan": NearestCentroid(metric="manhattan"),
    "RandomForest": RandomForestClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "KNeighbors": KNeighborsClassifier(),
    "SVC": OneVsRestClassifier(SVC(random_state=42)),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "Ridge": RidgeClassifier(random_state=42),
    "Perceptron": OneVsRestClassifier(Perceptron(random_state=42))
}

In [19]:
cross_val_prediction_dict_path = data_output_dir / "cross_val_prediction_dict.pkl"

def _run_prediction_with_save(vector_df, prediction_func, analysis_key_tuple):
    cross_val_prediction_dict = load_object(cross_val_prediction_dict_path)
    
    if analysis_key_tuple in cross_val_prediction_dict:
        return cross_val_prediction_dict[analysis_key_tuple]
    
    predicted_label_series = vector_df_to_predicted_label_series(vector_df, prediction_func=prediction_func).droplevel("approach")

    cross_val_prediction_dict = load_object(cross_val_prediction_dict_path)  # reload for parallelization
    cross_val_prediction_dict[analysis_key_tuple] = predicted_label_series
    save_object(cross_val_prediction_dict, cross_val_prediction_dict_path, overwrite=True)

    return predicted_label_series

def cv_predict_logreg_optimized(X, y):
    X_app = X.index.get_level_values("approach").unique()
    X_seg = X.index.get_level_values("segmenter").unique()

    assert (len(X_app)==1) and (len(X_seg)==1), "ERROR: Too many identifiers in input X"
    app, seg = X_app[0], X_seg[0]
    C_optimal = optimal_c_series[(app, seg)]

    clf = OneVsRestClassifier(LogisticRegression(penalty='l2', C=C_optimal, solver='liblinear', class_weight='balanced', max_iter=10000))
    pred_func = cv_predict_factory(clf, cv=LeaveOneOut())
    return pred_func(X, y)


In [20]:
if not cross_val_prediction_dict_path.exists():
    save_object({}, cross_val_prediction_dict_path, overwrite=True)
cross_val_prediction_dict = load_object(cross_val_prediction_dict_path)  # reload for parallelization

found_approaches = approach_vector_df.index.get_level_values("approach").unique()
job_dict = {(approach, clf_name, "LOO"): (approach_vector_df.loc[[approach]], cv_predict_factory(clf=classifier_store_dict[clf_name], cv=LeaveOneOut())) for approach, clf_name in product(found_approaches, classifier_store_dict.keys())}
[job_dict.update({(approach, "LogReg_Optimized", "LOO"): (approach_vector_df.loc[[approach]], cv_predict_logreg_optimized)}) for approach in found_approaches]

with tqdm_joblib(tqdm(desc="Classifier application", total=len(job_dict))) as progress_bar:
    Parallel(n_jobs=-1)(delayed(_run_prediction_with_save)(in_df, prediction_func=pred_func, analysis_key_tuple=key_tuple) for key_tuple, (in_df, pred_func) in job_dict.items())

Classifier application: 100%|██████████| 40/40 [00:00<00:00, 82.76it/s]


## Panel B

# Figure 3

18 x 9cm

# Figure 4

18 x 17cm

First row

    Panel a: 8 x 4
    Panel b: 4 x 4
    Panel c: 6 x 4

Second row

    Panel d: 8 x 4
    Panel e: 4 x 4
    Panel f: 6 x 4

Third row

    Panel g: 4 x 9
    Panel h: 14 x 9

## Panel A

## Panel B

## Panel C

## Panel D

## Panel E

## Panel F

## Panel G

## Panel H

# Figure 5

18 x 12cm

First row

    Panel a: 18 x 6
    Panel b: 18 x 6

## Panel A

In [21]:
drug_subsets = approach_scalars_df.loc["raw"].index.to_frame(index=False).groupby("drug_class")["drug"].unique().drop("control").to_dict()
drug_subsets = {k: list(v) + ["VEH"] for k,v in drug_subsets.items()}

for k, v in drug_subsets.items():
    _slice = approach_vector_df.loc["raw"].loc[pd.IndexSlice[segmenter_order, :, drug_subsets[k]], :]
    print(k, len(_slice)/2)

antidepressant 88.0
antipsychotic 169.0
benzo 89.0
snri 63.0
ssri 84.0
stimulant 108.0


In [22]:
subset_prediction_dict_path = data_output_dir / "subset_prediction_dict.pkl"
if not subset_prediction_dict_path.exists():
    save_object({}, subset_prediction_dict_path, overwrite=True)

def _run_prediction_with_save2(vector_df, prediction_func, analysis_key_tuple):
    for subset, subset_drugs in drug_subsets.items():
        subset_key_tuple = (subset, *analysis_key_tuple)

        subset_prediction_dict = load_object(subset_prediction_dict_path)        
        if subset_key_tuple in subset_prediction_dict:
            return subset_prediction_dict[subset_key_tuple]

        subset_vector_df = vector_df.loc[pd.IndexSlice[:, segmenter_order, :, subset_drugs], :]
        predicted_label_series = vector_df_to_predicted_label_series(subset_vector_df, prediction_func=prediction_func).droplevel("approach")

        subset_prediction_dict = load_object(subset_prediction_dict_path)        
        subset_prediction_dict[subset_key_tuple] = predicted_label_series
        save_object(subset_prediction_dict, subset_prediction_dict_path, overwrite=True)

    return predicted_label_series  # just return the last labels...

selected_classifiers = ["NC_Manhattan", "RandomForest", "SVC"]
job_dict = {("raw", clf_name, "LOO"): (approach_vector_df.loc[["raw"]], cv_predict_factory(clf=classifier_store_dict[clf_name], cv=LeaveOneOut())) for clf_name in selected_classifiers}
job_dict.update({("raw", "LogReg_Optimized", "LOO"): (approach_vector_df.loc[["raw"]], cv_predict_logreg_optimized)})


with tqdm_joblib(tqdm(desc="Classifier application", total=len(job_dict))) as progress_bar:
    Parallel(n_jobs=-1)(delayed(_run_prediction_with_save2)(in_df, prediction_func=pred_func, analysis_key_tuple=key_tuple) for key_tuple, (in_df, pred_func) in job_dict.items())

Classifier application: 100%|██████████| 4/4 [00:00<00:00,  8.35it/s]


## Panel B

In [23]:
rng_seed = 42
rng = np.random.default_rng(rng_seed)
non_vehicle_drugs = [d for d in drug_order if d != "VEH"]

l_step = 2
n_combinations = 15  # maximum up to l=14, l=15 is just 1

full_subset_dict = {}
for l in range(1, len(non_vehicle_drugs)+1, l_step):
    possible_combinations = list(combinations(non_vehicle_drugs, r=l))
    if len(possible_combinations) >= n_combinations:
        sampled_combinations = rng.choice(possible_combinations, size=n_combinations, replace=False)
    else:
        sampled_combinations = list(possible_combinations)
    sampled_combinations = [list(c) for c in sampled_combinations]
    for i, comb in enumerate(sampled_combinations):
        full_subset_dict[(l, i)] = ["VEH"] + comb
full_subset_dict

{(1, 0): ['VEH', 'alprazolam'],
 (1, 1): ['VEH', 'atomoxetine'],
 (1, 2): ['VEH', 'methamphetamine'],
 (1, 3): ['VEH', 'risperidone'],
 (1, 4): ['VEH', 'phenelzine'],
 (1, 5): ['VEH', 'clozapine'],
 (1, 6): ['VEH', 'methylphenidate'],
 (1, 7): ['VEH', 'bupropion'],
 (1, 8): ['VEH', 'modafinil'],
 (1, 9): ['VEH', 'chlorpromazine'],
 (1, 10): ['VEH', 'venlafaxine'],
 (1, 11): ['VEH', 'diazepam'],
 (1, 12): ['VEH', 'haloperidol'],
 (1, 13): ['VEH', 'fluoxetine'],
 (1, 14): ['VEH', 'citalopram'],
 (3, 0): ['VEH', 'alprazolam', 'chlorpromazine', 'citalopram'],
 (3, 1): ['VEH', 'diazepam', 'bupropion', 'methamphetamine'],
 (3, 2): ['VEH', 'clozapine', 'risperidone', 'venlafaxine'],
 (3, 3): ['VEH', 'haloperidol', 'venlafaxine', 'fluoxetine'],
 (3, 4): ['VEH', 'phenelzine', 'fluoxetine', 'methylphenidate'],
 (3, 5): ['VEH', 'bupropion', 'haloperidol', 'risperidone'],
 (3, 6): ['VEH', 'bupropion', 'clozapine', 'fluoxetine'],
 (3, 7): ['VEH', 'diazepam', 'clozapine', 'haloperidol'],
 (3, 8): ['

In [24]:
tmp_dir = data_output_dir / "tmp"
tmp_dir.mkdir(parents=True, exist_ok=True)

def _run_prediction_with_save2(subset_key_tuple, subset_vector_df, prediction_func):
    save_object({subset_key_tuple: vector_df_to_predicted_label_series(subset_vector_df, prediction_func=prediction_func).droplevel("approach")}, tmp_dir / (str(subset_key_tuple) + ".pkl"), overwrite=True)

raw_vector_df = approach_vector_df.loc[["raw"]]
job_dict = {
    ("raw", "NC_Manhattan", "LOO"): (raw_vector_df, cv_predict_factory(clf=classifier_store_dict["NC_Manhattan"], cv=LeaveOneOut())),
    ("raw", "LogReg_Optimized", "LOO"): (raw_vector_df, cv_predict_logreg_optimized),
    ("raw", "RandomForest", "LOO"): (raw_vector_df, cv_predict_factory(clf=classifier_store_dict["RandomForest"], cv=LeaveOneOut())),
    ("raw", "SVC", "LOO"): (raw_vector_df, cv_predict_factory(clf=classifier_store_dict["SVC"], cv=LeaveOneOut())),
}

subset_prediction_dict_path = data_output_dir / "full_subset_prediction_dict.pkl"
if not subset_prediction_dict_path.exists():
    save_object({}, subset_prediction_dict_path, overwrite=True)
subset_prediction_dict = load_object(subset_prediction_dict_path)

existing_tmp_dict = {}
[existing_tmp_dict.update(load_object(p)) for p in tmp_dir.glob(".pkl")]
subset_prediction_dict.update(existing_tmp_dict)

subset_key_tuples_vector_df_pred_func_dict = {}
for subset, subset_drugs in full_subset_dict.items():    
    for analysis_key_tuple, (vector_df, prediction_func) in job_dict.items():
        subset_key_tuple = (*subset, *analysis_key_tuple)
        if subset_key_tuple not in subset_prediction_dict:
            subset_key_tuples_vector_df_pred_func_dict[subset_key_tuple] = (vector_df.loc[pd.IndexSlice[:, segmenter_order, :, subset_drugs], :], prediction_func)

with tqdm_joblib(tqdm(desc="Classifier application", total=len(subset_key_tuples_vector_df_pred_func_dict))) as progress_bar:
    Parallel(n_jobs=-1)(delayed(_run_prediction_with_save2)(subset_key_tuple, subset_vector_df, prediction_func) for subset_key_tuple, (subset_vector_df, prediction_func) in subset_key_tuples_vector_df_pred_func_dict.items())

Classifier application:  19%|█▉        | 82/424 [00:25<01:45,  3.24it/s]


KeyboardInterrupt: 

In [26]:
subset_prediction_dict_path = data_output_dir / "full_subset_prediction_dict.pkl"
if not subset_prediction_dict_path.exists():
    save_object({}, subset_prediction_dict_path, overwrite=True)
else:
    subset_prediction_dict = load_object(subset_prediction_dict_path)

existing_tmp_dict = {}
[existing_tmp_dict.update(load_object(p)) for p in tmp_dir.glob("*.pkl")]
subset_prediction_dict.update(existing_tmp_dict)
save_object(subset_prediction_dict, subset_prediction_dict_path, overwrite=True)